# INTERVENE-2 on Colab GPU

Runs the locked protocol `intervene2/PROTOCOL_v5.md` end to end: development leave-one-cohort-out, automatic model selection, one-time sealed test (TCGA-LUAD, GSE50081) and interaction discovery.

1. **Runtime → Change runtime type → GPU**.
2. **Runtime → Run all**. Approve the Google Drive prompt.
3. Every finished fit is saved to `MyDrive/intervene2/`. If the session disconnects, just **Run all** again: it resumes.

Free Colab stops after inactivity or about 12 hours; with Colab Pro+ background execution it keeps running with your computer off. Either way nothing finished is lost.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/intervene2'
import os; os.makedirs(OUT, exist_ok=True)
!nvidia-smi -L || echo 'No GPU: Runtime > Change runtime type > GPU'

In [ ]:
%cd /content
!test -d intervene-luad-graph-attention && (cd intervene-luad-graph-attention && git pull -q) || git clone -q https://github.com/Kxvya18/intervene-luad-graph-attention.git
%cd /content/intervene-luad-graph-attention
!git log -1 --format='code version: %h %cd'
!pip -q install lifelines

In [ ]:
# Full run (resumable). Progress is printed and also appended to MyDrive/intervene2/run.log
!python -m intervene2.run --out "$OUT" --stage all 2>&1 | grep -v Warning | tee -a "$OUT/run.log"

In [ ]:
# Results summary
import pandas as pd, json
for f in ['dev_summary.csv','sealed_results.csv','sealed_pooled.csv','discovery_interactions_sealed.csv']:
    p = f'{OUT}/{f}'
    if os.path.exists(p):
        print('==', f); print(pd.read_csv(p).round(4).to_string(index=False))
for f in ['selection.json','sealed_adjusted.json','attention_stability.json']:
    p = f'{OUT}/{f}'
    if os.path.exists(p): print('==', f, open(p).read())

In [ ]:
# Zip everything (except the data cache) for download
!cd /content/drive/MyDrive && zip -qr intervene2_results.zip intervene2 -x 'intervene2/cache/*' 'intervene2/data.pkl' && ls -lh intervene2_results.zip